# Baseline 01 — Classical ML Defect Classification

This notebook establishes the first interpretable baseline for FactoryGuard AI.

Scope for this baseline:
- dataset inspection and EDA
- train/dev/test split
- random-guess baseline
- logistic regression baseline (flattened pixel features)
- evaluation via Accuracy, Macro F1, Confusion Matrix
- misclassification analysis and hypotheses for next experiments

> Deep learning is intentionally out of scope for Baseline 01.

Check environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import sklearn

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set(style='whitegrid')

## 1) Check data in raw
Expected structure under with one folder per class.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/raw/NEU-DET")

for path in DATA_DIR.rglob("*"):
    if path.is_file():
        print(path)
        break

for item in DATA_DIR.iterdir():
    print(item)

for split in ["train", "validation"]:
    split_dir = DATA_DIR / split

    print(f"\n===== {split.upper()} =====")

    for item in split_dir.iterdir():
        print(item)

..\data\raw\NEU-DET\validation\annotations\crazing_240.xml
..\data\raw\NEU-DET\train
..\data\raw\NEU-DET\validation

===== TRAIN =====
..\data\raw\NEU-DET\train\annotations
..\data\raw\NEU-DET\train\images

===== VALIDATION =====
..\data\raw\NEU-DET\validation\annotations
..\data\raw\NEU-DET\validation\images


## 2) Count images in each class

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/raw/NEU-DET")

TRAIN_IMG_DIR = DATA_DIR / "train" / "images"
VAL_IMG_DIR = DATA_DIR / "validation" / "images"

# rglob tìm file kể cả trong các folder con
train_images = list(TRAIN_IMG_DIR.rglob("*.jpg"))
val_images = list(VAL_IMG_DIR.rglob("*.jpg"))

print("Train images:", len(train_images))
print("Validation images:", len(val_images))
print("Total:", len(train_images) + len(val_images))

print("\nSome train images:")

for path in train_images[:20]:
    print(path)

Train images: 1440
Validation images: 360
Total: 1800

Some train images:
..\data\raw\NEU-DET\train\images\crazing\crazing_1.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_10.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_100.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_101.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_102.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_103.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_104.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_105.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_106.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_107.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_108.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_109.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_11.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_110.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_111.jpg
..\data\raw\NEU-DET\train\images\crazing\crazing_112.jpg
..\data\raw\NEU-DE

In [ ]:
classes = sorted([
    folder.name
    for folder in TRAIN_IMG_DIR.iterdir()
    if folder.is_dir()
])

print("Classes:")
for i, class_name in enumerate(classes):
    print(f"{i}: {class_name}")

print("\nNumber of classes:", len(classes))

Classes:
0: crazing
1: inclusion
2: patches
3: pitted_surface
4: rolled-in_scale
5: scratches

Number of classes: 6


## 3) Class counts and class distribution

In [ ]:
class_counts = {}

for class_name in classes:
    class_dir = TRAIN_IMG_DIR / class_name
    
    count = len(list(class_dir.glob("*.jpg")))
    
    class_counts[class_name] = count

print("Train:")
for class_name, count in class_counts.items():
    print(f"{class_name:20s}: {count}")


print("\nValidation:")

for class_name in classes:
    class_dir = VAL_IMG_DIR / class_name
    
    count = len(list(class_dir.glob("*.jpg")))
    
    print(f"{class_name:20s}: {count}")

Train:
crazing             : 240
inclusion           : 240
patches             : 240
pitted_surface      : 240
rolled-in_scale     : 240
scratches           : 240

Validation:
crazing             : 60
inclusion           : 60
patches             : 60
pitted_surface      : 60
rolled-in_scale     : 60
scratches           : 60


### Create metadata

In [ ]:
import pandas as pd

records = []

for split, split_dir in [
    ("train", TRAIN_IMG_DIR),
    ("validation", VAL_IMG_DIR)
]:
    for class_name in classes:
        class_dir = split_dir / class_name
        
        for image_path in class_dir.glob("*.jpg"):
            records.append({
                "path": str(image_path),
                "split": split,
                "class": class_name
            })

df = pd.DataFrame(records)

print("Shape:", df.shape)
df.head()

Shape: (1800, 3)


,path,split,class
0,..\data\raw\NEU-DET\train\images\crazing\crazi...,train,crazing
1,..\data\raw\NEU-DET\train\images\crazing\crazi...,train,crazing
2,..\data\raw\NEU-DET\train\images\crazing\crazi...,train,crazing
3,..\data\raw\NEU-DET\train\images\crazing\crazi...,train,crazing
4,..\data\raw\NEU-DET\train\images\crazing\crazi...,train,crazing


In [ ]:

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   path    1800 non-null   object
 1   split   1800 non-null   object
 2   class   1800 non-null   object
dtypes: object(3)
memory usage: 42.3+ KB
None


In [ ]:
# Check missing value
print("\nMissing values:")
print(df.isnull().sum())


Missing values:


NameError: name 'df' is not defined

## 4) Visualize representative images

In [ ]:
labels = sorted(df['label'].unique())
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for ax, lbl in zip(axes, labels):
    sample_path = df[df['label'] == lbl]['path'].iloc[0]
    img = Image.open(sample_path)
    ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
    ax.set_title(lbl)
    ax.axis('off')

for ax in axes[len(labels):]:
    ax.axis('off')

plt.suptitle('Representative images by class', y=1.02)
plt.tight_layout()
plt.show()

## 5) Define problem and metrics

- Multi-class defect classification
- Metrics: Accuracy, Macro F1, Confusion Matrix

## 6) Prepare flattened image features and split train/dev/test

In [ ]:
# Convert images to a common size and grayscale for classical baseline
IMG_SIZE = (64, 64)

X = []
y = []
for _, row in df.iterrows():
    with Image.open(row['path']) as img:
        img = img.convert('L').resize(IMG_SIZE)
        arr = np.array(img, dtype=np.float32).reshape(-1) / 255.0
    X.append(arr)
    y.append(row['label'])

X = np.array(X)
y = np.array(y)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_

# 70/15/15 split using stratification
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=SEED, stratify=y_encoded
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print('Train size:', len(y_train))
print('Dev size:  ', len(y_dev))
print('Test size: ', len(y_test))

## 7) Random-guess baseline

In [ ]:
dummy = DummyClassifier(strategy='stratified', random_state=SEED)
dummy.fit(X_train, y_train)

y_dev_dummy = dummy.predict(X_dev)

dummy_acc = accuracy_score(y_dev, y_dev_dummy)
dummy_f1 = f1_score(y_dev, y_dev_dummy, average='macro')
print(f'Random baseline (dev) Accuracy: {dummy_acc:.4f}')
print(f'Random baseline (dev) Macro F1: {dummy_f1:.4f}')

## 8) Logistic Regression baseline

In [ ]:
logreg = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(
        max_iter=2000,
        multi_class='multinomial',
        solver='lbfgs',
        random_state=SEED,
        n_jobs=None,
    )
)

logreg.fit(X_train, y_train)

y_dev_pred = logreg.predict(X_dev)

dev_acc = accuracy_score(y_dev, y_dev_pred)
dev_f1 = f1_score(y_dev, y_dev_pred, average='macro')
print(f'LogReg (dev) Accuracy: {dev_acc:.4f}')
print(f'LogReg (dev) Macro F1: {dev_f1:.4f}')

print('
Classification report (dev):')
print(classification_report(y_dev, y_dev_pred, target_names=class_names, zero_division=0))

## 9) Confusion Matrix

In [ ]:
cm = confusion_matrix(y_dev, y_dev_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression Confusion Matrix (Dev)')
plt.tight_layout()
plt.show()

## 10) Visualize misclassified examples

In [ ]:
mis_idx = np.where(y_dev != y_dev_pred)[0]
print('Number of misclassified dev samples:', len(mis_idx))

n_show = min(12, len(mis_idx))
if n_show == 0:
    print('No misclassifications to visualize.')
else:
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.flatten()

    for ax_i, idx in enumerate(mis_idx[:n_show]):
        ax = axes[ax_i]
        img = X_dev[idx].reshape(IMG_SIZE)
        true_lbl = class_names[y_dev[idx]]
        pred_lbl = class_names[y_dev_pred[idx]]
        ax.imshow(img, cmap='gray')
        ax.set_title(f'True: {true_lbl}
Pred: {pred_lbl}')
        ax.axis('off')

    for ax in axes[n_show:]:
        ax.axis('off')

    plt.suptitle('Misclassified Dev Examples', y=1.02)
    plt.tight_layout()
    plt.show()

## 11) Optional final test-set evaluation

Run this only after selecting the baseline configuration from dev results.

In [ ]:
# Uncomment after model selection is complete:
# y_test_pred = logreg.predict(X_test)
# print('LogReg (test) Accuracy:', accuracy_score(y_test, y_test_pred))
# print('LogReg (test) Macro F1:', f1_score(y_test, y_test_pred, average='macro'))

## 12) Observations and hypotheses

Use this section to record:
- What was changed and why
- Current metrics
- Common failure patterns from confusion matrix/misclassifications
- Hypotheses for next experiments (e.g., HOG features, regularization, data quality checks)